### Configure API Keys

In [ ]:
import os
from getpass import getpass

# Load environment variables from .env file if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Check if the required API keys are set in the environment variables, 
# if not prompt the user to enter them
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass("Enter your GROQ API key: ")

if not os.getenv('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass("Enter your Tavily API key: ")

if not os.getenv('PINECONE_API_KEY'):
    os.environ['PINECONE_API_KEY'] = getpass("Enter your Pinecone API key: ")

# Print the status of the API keys
print('GROQ_API_KEY configured:', bool(os.getenv('GROQ_API_KEY')))
print('TAVILY_API_KEY configured:', bool(os.getenv('TAVILY_API_KEY')))
print('PINECONE_API_KEY configured:', bool(os.getenv('PINECONE_API_KEY')))

### Load Real Docs

We will create our private knowledge base from a real web document:

for example:

https://docs.langchain.com/oss/python/langgraph/agentic-rag

This becomes our internal/private KB for the demo.

The web fallback is separate. That means:

-> First source: indexed LangGraph documentation page

-> Fallback source: live web search through Tavily

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

# Load documents from the specified URL using WebBaseLoader
loader = WebBaseLoader(
    web_path=(SOURCE_URL,),
    requests_kwargs={
        'headers': {
            'User-Agent': 'Mozilla/5.0 Agentic-RAG-Industry-Demo'
        }
    }
)

raw_docs = loader.load()

print('Loaded documents', len(raw_docs))
print('Source: ', raw_docs[0].metadata.get('source'))
print('\nPreview: \n')
print(raw_docs[0].page_content[:1500])

### Split Documents into Chunks

In [ ]:
from langchain_text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

print('Total chunks created:', len(chunks))
print('Preview of first chunk:\n')
print(chunks[0].page_content[:900])

### Create Free Local Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print('Embedding dimension:', len(sample_vector))

### Create the Pinecone Vector Database

In [ ]:
import time
from pinecone import Pinecone, SeverlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

# Create the index only if it does not already exist
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,       #all MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=SeverlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # wait until Pinecone reports the new index as ready
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print("Waiting for index to be ready...")
        time.sleep(5)

print(f"Index '{INDEX_NAME}' is ready.")

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready.")

In [ ]:
# Load Existing Index
from pinecone import Pinecone, SeverlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

# Load the existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding_function=embeddings.embed_query,
    namespace=NAMESPACE,
)

# Create a retriever from the vectorstore
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Existing Pinecone index loaded:", INDEX_NAME)

### Test Private KB Retrieval

In [ ]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f'\n --- KB Result {i} ---')
    print('Source:', doc.metadata.get('source'))
    print('Content Preview:', doc.page_content[:700])

### Initialize LLM

In [ ]:
from langchain_groq import ChatGroq

# Replace this model if ur Groq dashboard shows a different available model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

print(llm.invoke("Explain Agentic RAG in one sentence.").content)

### Initialize Tavily Web Search

It will be used only when the private knowledge base is insufficient

In [ ]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=True,
)

print("Tavily web search is ready.")

### Define Structured Decisions

Structure outputs help avoid parsing messy natural language.

In [ ]:
from typing import List, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document

class RouteDecision(BaseModel):
    route: Literal["kb", "direct"] = Field(
        description="Use kb for questions needing Agentic RAG docs; direct for greetings/simple chat."
    )

class EvidenceGrade(BaseModel):
    grade: Literal["good", "weak"] = Field(
        description="good means evidence can answer the questions; weak means not enough evidence."
    )


class AgentState(TypeDict):
    question: str
    current_querry: str
    kb_docs: List[Document]
    web_results: str
    kb_grade: str
    web_grade: str
    answer: str
    source_used: str
    retry_count: int